Run before everything

In [10]:
from ugot import ugot  # UGOT robot SDK
from IPython.display import clear_output  # Jupyter display helpers

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("192.168.88.1")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode"]
)

# Open UGOT camera for later image recognition
got.open_camera()

192.168.88.1:50051


Part2

In [ ]:
import time

# --- Global Tuning ---
# Positive Z (25) = Left | Negative Z (-25) = Right
# KP = Sensitivity (0.47 is high/sharp)
# BASE_SPEED = 22

def follow_line_step(kp, base_speed):
    """Calculates one frame of line following and returns all sensor data."""
    offset, line_type, x, y = got.get_single_track_total_info()
    
    # If line is completely lost
    if line_type == 0:
        return None, 0, 0, 0 
        ,# Proportional steering logic
    steering = int(offset * kp)
    speed_reduction = abs(int(steering * 0.25))
    current_speed = max(base_speed - speed_reduction, 12)
    
    got.mecanum_move_xyz(0, current_speed, steering)
    return offset, line_type, x, y

def part2run():
    print("Mission Started: Phase 1 (Initial Following)")
    got.set_track_recognition_line(0) 
    
    # --- PHASE 1: Follow line until Intersection ---
##     while True:
##         offset, line_type, x, y = follow_line_step(kp=0.45, base_speed=18)
##         intens = 5
##         if line_type == 0:
##             time.sleep(0.1)
##             if line_type == 1:
##                 got.mecanum_move_speed_times(direction=1, speed=20, times=0.5, unit=0)
##                 intens = 5
##                 got.mecanum_stop()
##                 break
##             got.mecanum_translate_speed(angle=90, speed=intens)
##             if line_type == 1:
##                 got.mecanum_move_speed_times(direction=1, speed=20, times=0.5, unit=0)
##                 intens = 5
##                 got.mecanum_stop()
##                 break
##             time.sleep(1)
##             got.mecanum_translate_speed(angle=-90, speed=intens)
##             if line_type == 1:
##                 got.mecanum_move_speed_times(direction=1, speed=20, times=0.5, unit=0)
##                 intens = 5
##                 got.mecanum_stop()
##                 break
##             time.sleep(1)
##             intens+5
##             print(intens)
##        time.sleep(0.01)

    # --- PHASE 2: Fuzzy OCR Detection ---
    choice = ""
    print("Scanning for text (Fuzzy Matching)...")
    
    while True:
        word = str(got.get_words_result()).upper()
        
        is_left = any(part in word for part in ["LEFT", "LEF", "EFT", "LFT"])
        is_right = any(part in word for part in ["RIGHT", "RIG", "GHT", "RGT"])
        
        if is_left:
            choice = "LEFT"
            got.screen_display_background(6) 
            print("Direction: LEFT")
            break
        elif is_right:
            choice = "RIGHT"
            got.screen_display_background(6)
            print("Direction: RIGHT")
            break
        
        got.mecanum_stop()
        time.sleep(0.1)

    # --- PHASE 3: The Directional Turn ---
    time.sleep(0.5)
    print(f"Executing {choice} turn...")
    
    # Step A: The Blind Nudge (Clear the intersection)
    z_move = 25 if choice == "LEFT" else -25
    got.mecanum_move_xyz(0, 15, z_move)
    time.sleep(0.8) 

    # Step B: The Acquisition Spin (Look for the new line)
    print("Spinning to lock onto new path...")
    searching = True
    while searching:
        # Use a slower pivot to avoid overshooting
        pivot_z = 18 if choice == "LEFT" else -18
        got.mecanum_move_xyz(0, 10, pivot_z)
            
        # IMPORTANT: Refresh sensor data inside this loop
        offset, line_type, x, y = got.get_single_track_total_info()
        
        # If line is found and is relatively centered
        if line_type != 0 and abs(offset) < 35:
            print("Line Locked!")
            searching = False
        time.sleep(0.01)

    # --- PHASE 4: Final Line Trace to Goal ---
    print("Phase 4: Following to Finish")
    while True:
        offset, line_type, x, y = follow_line_step(kp=0.45, base_speed=18)
        
        # Check for end-of-path markers or total line loss
        if line_type == 0:
            # Debounce: wait a moment to see if line returns
            time.sleep(0.2)
            _, check_lt, _, _ = got.get_single_track_total_info()
            if check_lt == 0:
                got.mecanum_stop()
                got.screen_print("FINISHED")
                print("Goal Reached.")
                break
        time.sleep(0.005)

# --- Execution ---
if __name__ == "__main__":
    try:
        part2run()
    except KeyboardInterrupt:
        got.mecanum_stop()
    except Exception as e:
        print(f"System Error: {e}")
        got.mecanum_stop()

Mission Started: Phase 1 (Initial Following)
5
5
5
